# Class activation Map with pyTorch.

The Class Activation Maps (CAM) method is one of the tools that help make models more understandable. CAM lets you visualise which parts of the input data (mostly images) are the most important for a particular prediction of the model. Like other activation maps, the method highlights the regions that influence the model's decision, and in this way provides visual explanations.

In this practice we will study how to apply the CAM method to interpret deep learning models, in particular convolutional neural networks (CNN), and we will also look at using CAM to highlight counterfactual regions.

**Goals of the practice:**

- Understand the implementation of the Class Activation Maps method.
- Learn to apply this method to visualise the important regions of the input data.
- Explore examples of using CAM to interpret the model's predictions.
- Consider using CAM to highlight counterfactual regions.
<img src="https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/assets/hartono-creative-studio-1g-W-pze-XX2-E-unsplash.jpg" alt="hartono-creative-studio-1g-W-pze-XX2-E-unsplash" border="0">

In [ ]:
from PIL import Image
from matplotlib.pyplot import imshow
import matplotlib.pyplot as plt

import torchvision.models as models
from torchvision import transforms

from torch.autograd import Variable
from torch.nn import functional as F
from torch import topk
import torch
import urllib
import requests
import numpy as np
from io import BytesIO

import numpy as np
import skimage.transform
from torchvision.models import resnet50, ResNet50_Weights, vgg16, VGG16_Weights

Let us load the image we are going to work with.

In [ ]:
# Loading the image
url = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat_and_dog.jpg'

image_bytes = requests.get(url).content
image = Image.open(BytesIO(image_bytes)) # Again we take one particular example x_0


plt.figure(figsize=(8,10))
plt.axis('off')
plt.imshow(image);

Let us define two pipelines:

- one for preprocessing the image before feeding it to the model (we take the same normalisation values as in the previous practice, since we are again working with a network trained on imagenet)

- one for keeping the original image, since we are going to compare the activation map with the original

In [ ]:
#1
preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
   transforms.Resize((224,224)),
])

#2
display = transforms.Compose([transforms.Resize((224,224))])

## **Building a Class Activation Map.**

Let us recall the process of building a CAM from the lesson. For it we need:

-  The activation maps of a convolutional layer (usually the last one). $A^k \in R^{u\times v}$, $k=1,2 ... n$
- The weights responsible for the prediction of class $c$ ($w^c)$.

Given these variables, the activation map is written as:
$$M_c(x, y) = \sum_k w^c_kA_k $$ — the class activation map for class $c$.

​How do we get the activation maps $A_k$ of the last layer? ***Hooks*** will help us here.

So, we need to extract the activation map of the last layer from the model.

Using "hooks" (*Hooks*) — literally "hooks" or "interceptors" — will help us solve this task.

The notion of a *Hook* is not specific to DL and is also found in other areas of programming.

**In general terms one can put it like this:**

**Hooks** are functions that are executed automatically after a certain event.

In the context of pyTorch, hooks are a mechanism for getting information about the behaviour of neural networks during the forward and the backward pass. They let you attach custom functions (which are the hooks themselves) to tensors and to modules (`torch.nn.Module`) of a neural network, making it possible to track, modify or record various aspects of the computational graph.

We will use a Hook to get the activation map from the last convolutional layer, but first let us look at a simple example.

**Consider the function:** $$f=u^2=(a^2+b)^2,$$
at $a=2, b=3$.

Let us do a *Forward pass* — simply compute the function at the given $a$ and $b$:

$f=u^2=(a^2+b)^2= (2^2+3)^2 = 49$

After that, let us compute the *Backward pass*. Recall that it is computed by the "chain" rule ([chain rule](https://en.wikipedia.org/wiki/Chain_rule)).

$\frac{\partial f}{\partial f}=1$


$\frac{\partial f}{\partial a} = \frac{\partial f}{\partial f}\frac{\partial f}{\partial u}\frac{\partial u}{\partial a} = 1*2u*2a=1*2*7*2*2=14*4=56$

$\frac{\partial f}{\partial b} = \frac{\partial f}{\partial f}\frac{\partial f}{\partial u}\frac{\partial u}{\partial b} = 1*2u*1=2*7*1=14=14$

To keep the gradients of all the parameters of the function we use the `.retain_grad()` method. By default PyTorch discards the gradients of intermediate tensors in order to save memory. The `.retain_grad()` method lets you say explicitly that the gradients must be kept for intermediate tensors as well.

In [ ]:
# Let us define two tensors and a third one as a function of them
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = (a**2 + b)**2

c.retain_grad() # we tell pyTorch to keep the gradients for intermediate tensors
c.backward()

print(c)

print(a.grad)
print(b.grad)
print(c.grad)

Now let us attach a simple "hook" to the backward pass of our graph. We will add 2 to the gradient we get.

In [ ]:
# Let us define the hook function
def hook(grad):
  return grad + 2

a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = (a**2 + b)**2

c.retain_grad()
c.register_hook(hook)

c.backward()

print(c)

print(a.grad)
print(# Your code here)
print(c.grad)

**Quiz** What is the gradient of the parameter $b$ equal to after the hook has been added?

**How the output values are obtained.**

Consider the same function $f=u^2=(a^2+b)^2$, at $a=2, b=3$

**Forward pass:**

$f=u^2=(a^2+b)^2= (2^2+3)^2 = 49$

**Backward pass + Hook:**

$\frac{\partial f}{\partial f}=1 + 2=3$


$\frac{\partial f}{\partial a} = \frac{\partial f}{\partial f}\frac{\partial f}{\partial u}\frac{\partial u}{\partial a} = 3*2u*2a=3*2*7*2*2=3*14*4=168$

#### **A Hook for the forward pass**

Now let us apply a Hook to the forward pass of the model we are considering. For convenience, we will write the **Hook** as a class.

In [ ]:
class Hook():

    def __init__(self, m):
      self.hook = m.register_forward_hook(self.hook_func) #we hook onto the forward pass of the layer m passed at initialisation

    def hook_func(self, module, input, output):
      self.features = ((output.cpu()).data).numpy() #we take the output of the layer and turn it into a numpy array

    def remove(self):
      self.hook.remove()

Let us load the model. We will use ResNet50 trained on imagenet.

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1) # loading the model
model.eval();

And let us prepare the image to feed it to the neural network.

In [ ]:
preprocessed_image = preprocess(image).unsqueeze(0) # Preprocessing and adding the batch dimension

Now let us attach our hook. To do that we will find and extract the last layer.

**Quiz** Look at the architecture of the model. What is the name of the last layer **before** Average Pooling?

In [ ]:
#model #layer4

Now let us extract this layer and attach the hook to it.

In [ ]:
final_layer = model._modules.get('layer4')
act_maps = Hook(final_layer) # we hook onto layer 4

In [ ]:
vgg_final_layer = vgg.features[-1]
vgg_act_maps = Hook(vgg_final_layer)

At this point our **Hook** has been initialised but has not intercepted anything yet. To get the map we are interested in, we do a Forward pass.

Also, since the last layer of the model was simply a linear layer, we then push the values through softmax and extract the probabilities.

In [ ]:
prediction = model(preprocessed_image) #forward pass
pred_prob = F.softmax(prediction, -1).data.squeeze()

Let us remove the Hook, as we do not need it any more.

In [ ]:
act_maps.remove()

**Quiz** What does the output of the last convolutional layer — the activation map at the last layer before AdaptiveAvgPool — correspond to?

**Hint:** using `torch.summary(model, input_shape)` may help you.

In [ ]:
!pip install -q torchinfo


In [ ]:
from torchinfo import summary
# Your code here

Now we need to extract the weights *associated with our prediction*. For this we will need:
- the prediction itself;
- the parameters of the very last (fully connected) layer;

In [ ]:
#Getting the prediction

idx = topk(pred_prob, 1)[1].int()[0]

print(f'The model predicts class no. {idx}')

Class 179 corresponds to Staffordshire bull terrier.  It looks like the model was not wrong! Let us see who made it into the top 3.

In [ ]:
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
urllib.request.urlretrieve(url, "imagenet_classes.txt")

with open("imagenet_classes.txt", "r") as f:
    categories = [s.strip() for s in f.readlines()]

In [ ]:
for i in topk(pred_prob, 10)[1]:
  print(categories[i])

Let us extract the parameters of the very last layer. Note that their shape is (`num_classes`, `num_in_features`), since every class has its own set of weights that links the input feature maps to that class.

In [ ]:
last_layer_params = list(model._modules.get('fc').parameters())[0]

In [ ]:
weights_interested = model.fc.weight[245, :]  #w^c_k we extract the class we are interested in
weights_interested = weights_interested.cpu().data.numpy()

From which, by the formula
$$M_c(x, y) = \sum_k w^c_kA_k $$

​

In [ ]:
cam = weights_interested.dot(act_maps.features.reshape((2048, 7*7))) #elementwise product
print(cam.shape)

And let us bring the feature map into a readable form.

In [ ]:
cam = cam.reshape((7,7))

plt.imshow(cam, alpha=0.5, cmap='jet');
plt.axis('off');

To overlay the feature map on the image, we need to bring it to the corresponding size. Namely, from $7\times 7$ to $224\times 224$. This can be done with `torch.nn.Functional.interpolate()`.

In [ ]:
cam_tensor = torch.from_numpy(cam).float().unsqueeze(0).unsqueeze(0)  # Converting back into a tensor
interpol = F.interpolate(cam_tensor, (224, 224), mode="bilinear").squeeze(0).squeeze(0) # Interpolating the map onto the image

plt.figure(figsize=(8,8))
plt.subplot(221)
plt.imshow(interpol, alpha=0.5, cmap='jet');
plt.title('CAM')
plt.axis('off')

To finish, let us make it pretty and display the result.

In [ ]:
plt.figure(figsize=(8,8))
plt.subplot(221)
plt.imshow(display(image))
plt.axis('off')
plt.title('Original Image')

plt.subplot(222)
imshow(preprocessed_image.squeeze(0).permute(1, 2, 0))
plt.axis('off')
plt.title('Image Tensor')


plt.subplot(223)
imshow(display(image))
plt.axis('off')
imshow(interpol, alpha=0.5, cmap='jet');
plt.title('Original + CAM')


plt.subplot(224)
imshow(preprocessed_image.squeeze(0).permute(1, 2, 0))
plt.axis('off')
imshow(interpol, alpha=0.5, cmap='jet')
plt.title('Tensor + CAM');

## Counterfactual regions with  CAM.

In the theory part we showed that taking the antigradient (all the activation maps with a "-" sign) can help find counterfactual regions. In the context of CV this means that we can see the regions such that focusing attention on them would lead to a different prediction. However, this is not always the case, and a simply negative activation map may turn out to be useless.

For example, if we take the activation maps of the class Staffordshire bullterrier (the predicted one), we will simply get a reflection of the region.

In [ ]:
#See for yourself by running the code

weights_interested = model.fc.weight[idx, :]  #w^c_k we extract the class we are interested in
weights_interested = weights_interested.cpu().data.numpy()

negative_cam = weights_interested.dot(-act_maps.features.reshape((2048, 7*7))) #elementwise product with the "-" sign

negative_cam = negative_cam.reshape((7,7))

negative_cam_tensor = torch.from_numpy(negative_cam).float().unsqueeze(0).unsqueeze(0)  # Add two dimensions: channel and batch
interpol = F.interpolate(negative_cam_tensor, (224, 224), mode="bilinear")

plt.figure(figsize=(8,8))
plt.subplot(221)
plt.imshow(interpol.squeeze(0).squeeze(0), alpha=0.5, cmap='jet');
plt.title('CAM')
plt.axis('off')

However, this does not mean that the theory itself does not work. Let us look at the counterfactual explanation for class 283 - Persian cat.

In [ ]:
cat_weights_interested = model.fc.weight[283, :]  #w^c_k we extract the class we are interested in
cat_weights_interested = cat_weights_interested.cpu().data.numpy()

negative_cat_cam = cat_weights_interested.dot(-act_maps.features.reshape((2048, 7*7))) #elementwise product with the "-" sign

negative_cat_cam = negative_cat_cam.reshape((7,7))

negative_cat_cam_tensor = torch.from_numpy(negative_cat_cam).float().unsqueeze(0).unsqueeze(0)  # Add two dimensions: channel and batch
interpol = F.interpolate(negative_cat_cam_tensor, (224, 224), mode="bilinear")

plt.figure(figsize=(8,8))
plt.subplot(221)
plt.imshow(interpol.squeeze(0).squeeze(0), alpha=0.5, cmap='jet');
plt.title('CAM')
plt.axis('off')

The theory works — the counterfactual object to a cat is a dog! What is the conclusion from this:

- the theoretical properties of explainable AI methods are not robust, but knowing them can help you extract additional insights from the model.